In [ ]:
# === Imports ===
import os
import pandas as pd
import matplotlib.pyplot as plt
from ipyfilechooser import FileChooser
from IPython.display import display
from matplotlib.lines import Line2D

# === Dossier de départ ===
dpath = "//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_data/L1NDNF_mice/NV/Allocentric_task/Training/2026_02_23/Cheeseboard/11-52-58_11-53-34/OpenEphys"
print("🔍 Sélectionnez votre fichier Excel ou CSV à analyser:")

# Sélecteur de fichier
fc1 = FileChooser(
    dpath,
    filter_pattern=['*.xlsx', '*.csv'],
    title="<b>Sélectionnez votre fichier Excel/CSV à analyser</b>"
)
display(fc1)

# === Callback pour traiter le fichier choisi ===
def update_file(chooser):
    global fichier_choisi, dpath
    fichier_choisi = chooser.selected
    dpath = os.path.dirname(fichier_choisi)
    get_ipython().run_line_magic('store', 'dpath')
    print(f"✅ Fichier sélectionné: {os.path.basename(fichier_choisi)}")
    
    # === Lecture du fichier ===
    if fichier_choisi.endswith(".csv"):
        df = pd.read_csv(fichier_choisi, sep=',')  # Ajout du séparateur ;
    else:
        df = pd.read_excel(fichier_choisi)

    print(df.columns) 
    
    print(f"\n📊 Aperçu des données:")
    print(df.head())
    print(f"\nNombre d'épisodes: {len(df)}")
    
    # === Les colonnes sont déjà en secondes (start_time, end_time) ===
    # Pas besoin de conversion
    
    # === Couleurs et positions verticales ===
    colors = {
        "Wake_1": "#8FE73C",  # vert clair
        "NREM_2": "#E01010",  # rouge
        "REM_3":  "#39C0E2",  # bleu
    }
    y_positions = {"Wake_1": 1.0, "Sleep": 0.0}
    
    fig, ax = plt.subplots(figsize=(17, 3))
    linewidth = 0.5  # largeur des lignes
    
    # --- Wake ---
    for _, row in df[df["label"] == "Wake_1"].iterrows():
        ax.plot([row["start_time"], row["end_time"]],
                [y_positions["Wake_1"], y_positions["Wake_1"]],
                color=colors["Wake_1"], linewidth=linewidth, solid_capstyle="butt", alpha=0.9)
    
    # --- Sleep : NREM_2 + REM_3 ---
    for lbl in ["NREM_2", "REM_3"]:
        for _, row in df[df["label"] == lbl].iterrows():
            ax.plot([row["start_time"], row["end_time"]],
                    [y_positions["Sleep"], y_positions["Sleep"]],
                    color=colors[lbl], linewidth=linewidth, solid_capstyle="butt", alpha=0.9)
    
    # === Mise en forme ===
    ax.set_yticks([y_positions["Wake_1"], y_positions["Sleep"]])
    ax.set_yticklabels(["Wake", "Sleep (NREM/REM)"])
    ax.set_xlabel("Temps (heures)")
    ax.set_title("Polysomnographe compact — Wake / NREM_2 & REM_3")
    ax.grid(True, axis="y", alpha=0.7)
    
    # === Axe X en heures, ticks toutes les 2 heures ===
    max_time = df["end_time"].max()
    hours = range(0, int(max_time // 3600) + 2, 2)
    ax.set_xticks([h*3600 for h in hours])
    ax.set_xticklabels([f"{h}h" for h in hours])
    
    ax.set_xlim(0, max_time + 60)  # padding 1 min
    ax.set_ylim(-0.5, 1.5)
    
    # === Légende ===
    legend_elements = [
        Line2D([0], [0], color=colors["Wake_1"], lw=8, label="Wake"),
        Line2D([0], [0], color=colors["NREM_2"], lw=8, label="NREM_2"),
        Line2D([0], [0], color=colors["REM_3"], lw=8, label="REM_3"),
    ]
    ax.legend(bbox_to_anchor=(1.05, 1), handles=legend_elements, loc="upper right")

    plt.tight_layout()
    plt.show()
    
    # === Enregistrement dans le même dossier que le fichier choisi ===
    folder = os.path.dirname(fichier_choisi)
    output_file = os.path.join(folder, "Hypnogramme.png")
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"✅ Figure enregistrée : {output_file}")

# Enregistrer le callback
fc1.register_callback(update_file)


Hypnogramm with vigilantes states and LFP Signal (to improve)

In [ ]:
# =========================================================
# IMPORTS
# =========================================================

import os
import json
from importlib_metadata import metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pathlib import Path
from matplotlib.patches import Rectangle

from scipy.signal import butter, filtfilt, decimate

from ipyfilechooser import FileChooser
from IPython.display import display

# =========================================================
# PARAMÈTRES UTILISATEUR
# =========================================================

# Canaux (index 0-based dans le fichier .dat)
CHANNELS = {
    "PFC": 0,
    "S1" : 1,
    "CA1": 2,
    "RSC": 3,
    "EMG": 4,
}

# Sample rate par défaut
SAMPLERATE_DEFAULT = 1000  # Hz

# Fenêtre temporelle
START_TIME_PLOT  = 0
DURATION_TO_PLOT = None

# Downsample affichage
DOWNSAMPLE = 5

# Couleurs signaux
SIGNAL_COLORS = {
    "PFC": "#2E8B57",
    "S1" : "#6A0DAD",
    "CA1": "#3A50D9",
    "RSC": "#DB19B1",
    "EMG": "black",
}

# Couleurs hypnogramme
HYPNO_COLORS = {
    "AW"    : "#1a1a1a",
    "AW_1"  : "#1a1a1a",
    "Wake"  : "#1a1a1a",

    "QW"    : "#808080",
    "QW_2"  : "#808080",

    "NREM"  : "#D0D0D0",
    "NREM_3": "#D0D0D0",

    "IS"    : "#A0A0A0",
    "IS_4"  : "#A0A0A0",

    "REM"   : "white",
    "REM_5" : "white",
}

# =========================================================
# DOSSIERS PAR DÉFAUT
# =========================================================

DPATH_SCORING = r"\\10.69.168.1\crnldata\forgetting\Aurelie\MiniscopeOE_data\L1NDNF_mice\NV\Allocentric_task\Training\2026_02_23\SleepBefore"

DPATH_DAT = r"\\10.69.168.1\crnldata\forgetting\Aurelie\MiniscopeOE_data\L1NDNF_mice\NV\Allocentric_task\Training\2026_02_23\SleepBefore"

# =========================================================
# STOCKAGE GLOBAL
# =========================================================

selected = {
    "csv": None,
    "dat": None,
}

# =========================================================
# FILECHOOSER CSV
# =========================================================

print("=" * 60)
print("ÉTAPE 1 — Sélectionnez le fichier de scoring")
print("=" * 60)

fc_csv = FileChooser(
    DPATH_SCORING,
    filter_pattern=["*.csv", "*.xlsx"],
    title="<b> Fichier de scoring</b>"
)

display(fc_csv)

# =========================================================
# FILECHOOSER DAT
# =========================================================

print("=" * 60)
print("ÉTAPE 2 — Sélectionnez le fichier .dat")
print("=" * 60)

fc_dat = FileChooser(
    DPATH_DAT,
    filter_pattern=["*.dat"],
    title="<b> Fichier .dat OpenEphys</b>"
)

display(fc_dat)

# =========================================================
# FILTRE BANDE
# =========================================================

def bandpass_filter(signal, fs, low, high, order=4):

    nyq = fs / 2

    b, a = butter(
        order,
        [low / nyq, high / nyq],
        btype="band"
    )

    return filtfilt(b, a, signal)

# =========================================================
# FONCTION PRINCIPALE
# =========================================================

def generate_figure():

    csv_path = selected["csv"]
    dat_path = selected["dat"]

    if csv_path is None or dat_path is None:
        print(" En attente des deux fichiers...")
        return

    print("\n" + "=" * 60)
    print(" Génération de la figure")
    print("=" * 60)

    print(f"CSV : {csv_path}")
    print(f"DAT : {dat_path}")

    # =====================================================
    # 1. CHARGEMENT SCORING
    # =====================================================

    if csv_path.endswith(".csv"):
        df_score = pd.read_csv(csv_path)
    else:
        df_score = pd.read_excel(csv_path)

    df_score.columns = df_score.columns.str.strip().str.lower()

    print("\n Colonnes détectées :", df_score.columns.tolist())

    if all(col in df_score.columns for col in ["time", "duration", "label"]):

        df_score["start_time"] = df_score["time"]
        df_score["end_time"]   = df_score["time"] + df_score["duration"]

    elif all(col in df_score.columns for col in ["start_time", "end_time", "label"]):

        pass

    elif all(col in df_score.columns for col in ["start", "end", "label"]):

        df_score["start_time"] = df_score["start"]
        df_score["end_time"]   = df_score["end"]

    else:

        raise ValueError(
            " Format scoring non reconnu.\n"
            f"Colonnes trouvées : {df_score.columns.tolist()}"
        )

    labels_uniques = df_score["label"].unique()

    print(" Labels détectés :", labels_uniques)

    # =====================================================
    # 2. SAMPLE RATE
    # =====================================================

    datfilepath = Path(dat_path)

    oebin_file = datfilepath.parent.parent.parent.parent.parent / "structure.oebin"

    samplerate = SAMPLERATE_DEFAULT
    numchannel = None
    bit_volts = 1.0

    if oebin_file.exists():

        print(f"\n structure.oebin trouvé")

        with open(oebin_file) as f:
            metadata = json.load(f)

        samplerate = int(metadata["continuous"][0]["sample_rate"])
        numchannel = metadata["continuous"][0]["num_channels"]

        bit_volts = metadata["continuous"][0]["channels"][0]["bit_volts"]

        print(f" Sample rate : {samplerate} Hz")
        print(f" Nb canaux   : {numchannel}")
        print(f" Conversion : {bit_volts} µV/count")

    else:

        print("\n structure.oebin introuvable")
        print(f"→ Sample rate défaut : {samplerate} Hz")

    # =====================================================
    # 3. CHARGEMENT DAT
    # =====================================================

    print("\n Chargement du .dat ...")

    raw = np.fromfile(dat_path, dtype="int16")

    if numchannel is None:

        duree_scoring = df_score["end_time"].max()

        for n in [32, 16, 8, 64]:

            if raw.size % n == 0:

                n_samples_test = raw.size // n
                duree_test = n_samples_test / samplerate

                if abs(duree_test - duree_scoring) < duree_scoring * 0.1:

                    numchannel = n
                    print(f"→ Nb canaux détecté : {numchannel}")
                    break

        if numchannel is None:

            numchannel = 32
            print(f"→ Nb canaux par défaut : {numchannel}")

    DataRec = raw.reshape(-1, numchannel)

    duree_reelle = DataRec.shape[0] / samplerate

    print(f" DAT chargé : {DataRec.shape}")
    print(f"⏱ Durée réelle : {duree_reelle:.1f} s")

    # =====================================================
    # 4. CHECK CANAUX
    # =====================================================

    for name, ch_idx in CHANNELS.items():

        if ch_idx >= DataRec.shape[1]:

            raise IndexError(
                f" Canal {name} hors limites"
            )

    # =====================================================
    # 5. EXTRACTION FENÊTRE
    # =====================================================

    dur = DURATION_TO_PLOT if DURATION_TO_PLOT is not None else duree_reelle

    duration_plot = min(dur, duree_reelle - START_TIME_PLOT)

    start_idx = int(START_TIME_PLOT * samplerate)
    end_idx   = int((START_TIME_PLOT + duration_plot) * samplerate)

    end_idx = min(end_idx, DataRec.shape[0])

    ds = DOWNSAMPLE

    print(f"\n Fenêtre affichée : {START_TIME_PLOT}s → {START_TIME_PLOT + duration_plot:.1f}s")

    # =====================================================
    # 6. FILTRAGE + DOWNSAMPLE
    # =====================================================

    signals_ds = {}

    for name, ch_idx in CHANNELS.items():

        sig = (
            DataRec[start_idx:end_idx, ch_idx]
            .astype(float)
            * bit_volts
        )

        # centrage robuste
        sig = sig - np.mean(sig)

        if name != "EMG":

            sig = bandpass_filter(
                sig,
                samplerate,
                0.5,
                100
            )

        else:

            sig = bandpass_filter(
                sig,
                samplerate,
                30,
                120
            )
        # =============================================
        # DECIMATION PROPRE
        # =============================================

        sig_ds = decimate(sig, ds, zero_phase=True)

        signals_ds[name] = sig_ds

    # =====================================================
    # TEMPS DOWNSAMPLED
    # =====================================================

    npoints = len(signals_ds["PFC"])

    time = np.linspace(
        START_TIME_PLOT,
        START_TIME_PLOT + duration_plot,
        npoints
    )

    print(f" Points affichés : {npoints}")

    # =====================================================
    # 7. FIGURE
    # =====================================================

    signal_names = list(CHANNELS.keys())

    n_sig = len(signal_names)

    fig, axes = plt.subplots(
        n_sig + 1,
        1,
        figsize=(20, 10),
        sharex=True,
        gridspec_kw={
            "height_ratios": [1] * n_sig + [0.35]
        }
    )

    fig.patch.set_facecolor("white")

    # =====================================================
    # 8. TRACE SIGNAUX
    # =====================================================

    for i, name in enumerate(signal_names):

        ax = axes[i]

        sig = signals_ds[name]

        # =================================================
        # NORMALISATION ROBUSTE
        # =================================================

        robust_std = np.percentile(np.abs(sig), 99.5)

        if robust_std == 0:
            robust_std = 1

        if name == "EMG":
            scale = 0.5

        elif name == "RSC":
            scale = 0.5

        elif name == "CA1":
            scale = 0.5

        elif name == "PFC":
            scale = 0.5

        elif name == "S1":
            scale = 0.5

        else:
            scale = 0.5

        sig_plot = sig / robust_std * scale


        ax.plot(
            time,
            sig_plot,
            color=SIGNAL_COLORS[name],
            linewidth=0.35,
            alpha=0.9,
            antialiased=True,
            rasterized=True
        )

        # =============================================
        # LABELS STYLE PAPER
        # =============================================

        if name == "PFC":
            label = "LFP PFC\nfilt. 10-18Hz"

        elif name == "S1":
            label = "LFP S1\nfilt. 10-18Hz"

        elif name == "CA1":
            label = "LFP CA1\nfilt. 5-9Hz"

        elif name == "RSC":
            label = "LFP RSC\nfilt. 3-12Hz"

        else:
            label = "EMG"

        ax.set_ylabel(
            label,
            rotation=0,
            fontsize=11,
            labelpad=50,
            weight="normal",
            color=SIGNAL_COLORS[name],
            va="center"
        )

        ax.set_yticks([])

        ax.set_facecolor("white")

        for spine in ax.spines.values():
            spine.set_visible(False)

    # =====================================================
    # 9. BARRE D'ÉCHELLE
    # =====================================================

    ax_last_sig = axes[n_sig - 1]

    scale_dur = 60

    x_scale = START_TIME_PLOT + duration_plot - 1.4 * scale_dur

    y_lim = ax_last_sig.get_ylim()

    y_scale = y_lim[0] + (y_lim[1] - y_lim[0]) * 0.15

    ax_last_sig.plot(
        [x_scale, x_scale + scale_dur],
        [y_scale, y_scale],
        color="black",
        linewidth=3,
        solid_capstyle="butt"
    )

    ax_last_sig.text(
        x_scale + scale_dur / 2,
        y_scale + (y_lim[1] - y_lim[0]) * 0.08,
        "1 min",
        ha="center",
        va="bottom",
        fontsize=10
    )

    # =====================================================
    # 10. HYPNOGRAMME
    # =====================================================

    ax_hyp = axes[-1]

    ax_hyp.set_facecolor("white")

    df_plot = df_score[
        (df_score["end_time"] >= START_TIME_PLOT) &
        (df_score["start_time"] <= START_TIME_PLOT + duration_plot)
    ].copy()

    # =====================================================
    # FUSION ÉTATS IDENTIQUES
    # =====================================================

    merged_states = []

    current_label = None
    current_start = None
    current_end   = None

    for _, row in df_plot.iterrows():

        label = str(row["label"]).strip()

        start = row["start_time"]
        end   = row["end_time"]

        if current_label is None:

            current_label = label
            current_start = start
            current_end   = end

        elif label == current_label:

            current_end = end

        else:

            merged_states.append({
                "label": current_label,
                "start": current_start,
                "end": current_end
            })

            current_label = label
            current_start = start
            current_end   = end

    merged_states.append({
        "label": current_label,
        "start": current_start,
        "end": current_end
    })

    # =====================================================
    # TRACE HYPNOGRAMME
    # =====================================================

    for state in merged_states:

        x0 = max(state["start"], START_TIME_PLOT)

        x1 = min(state["end"], START_TIME_PLOT + duration_plot)

        width = x1 - x0

        color = HYPNO_COLORS.get(state["label"], "red")

        rect = Rectangle(
            (x0, 0),
            width,
            1,
            facecolor=color,
            edgecolor="none"
        )

        ax_hyp.add_patch(rect)

    # =====================================================
    # LÉGENDE
    # =====================================================

    legend_order = [
        "AW",
        "QW",
        "NREM",
        "IS",
        "REM"
    ]

    legend_patches = []

    for lbl in legend_order:

        legend_patches.append(
            mpatches.Patch(
                facecolor=HYPNO_COLORS[lbl],
                edgecolor="black",
                linewidth=0.5,
                label=lbl
            )
        )

    ax_hyp.legend(
        handles=legend_patches,
        loc="lower center",
        ncol=len(legend_patches),
        fontsize=10,
        frameon=False,
        bbox_to_anchor=(0.5, -0.8)
    )

    ax_hyp.set_ylim(0, 1)

    ax_hyp.set_yticks([])

    for spine in ax_hyp.spines.values():
        spine.set_visible(False)

    ax_hyp.set_xlim(
        START_TIME_PLOT,
        START_TIME_PLOT + duration_plot
    )

    ax_hyp.set_xticks([])

    # =====================================================
    # TITRE
    # =====================================================

    session_name = Path(dat_path).parent.parent.parent.parent.parent.name

    fig.suptitle(
        f"{session_name}",
        fontsize=14,
        weight="bold",
        y=0.995
    )

    plt.tight_layout(h_pad=0.05)

    # =====================================================
    # SAUVEGARDE
    # =====================================================

    plt.savefig(os.path.join(os.path.expanduser("~/Documents"), "Hypnogramme_PaperStyle.svg"),
                format="svg", bbox_inches="tight", facecolor="white")
# =========================================================
# CALLBACKS
# =========================================================

def on_csv_selected(chooser):

    selected["csv"] = chooser.selected

    print(f" CSV sélectionné :")
    print(selected["csv"])

    generate_figure()

def on_dat_selected(chooser):

    selected["dat"] = chooser.selected

    print(f" DAT sélectionné :")
    print(selected["dat"])

    generate_figure()

fc_csv.register_callback(on_csv_selected)
fc_dat.register_callback(on_dat_selected)


Hypnogramm with vigilantes states and LFP Signal (GOOD VERSION)

In [ ]:
# =========================================================
# IMPORTS
# =========================================================

import os
import json
from importlib_metadata import metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from pathlib import Path
from matplotlib.patches import Rectangle

from scipy.signal import butter, filtfilt, decimate

from ipyfilechooser import FileChooser
from IPython.display import display

# =========================================================
# PARAMÈTRES UTILISATEUR
# =========================================================

CHANNELS = {
    "PFC": 0,
    "S1" : 1,
    "CA1": 2,
    "RSC": 3,
    "EMG": 4,
}

SAMPLERATE_DEFAULT = 1000  # Hz

START_TIME_PLOT  = 0
DURATION_TO_PLOT = None

DOWNSAMPLE = 5

SIGNAL_COLORS = {
    "PFC": "#2E8B57",
    "S1" : "#6A0DAD",
    "CA1": "#3A50D9",
    "RSC": "#DB19B1",
    "EMG": "black",
}

# =========================================================
# FILTRES PAR CANAL (Hz)
# =========================================================

BANDPASS = {
    "PFC": (1,  50),
    "S1" : (1,  50),
    "CA1": (1,  50),
    "RSC": (1,  50),
    "EMG": (30, 120),
}

# Couleurs hypnogramme
HYPNO_COLORS = {
    "AW"    : "#1a1a1a",
    "AW_1"  : "#1a1a1a",
    "Wake"  : "#1a1a1a",
    "QW"    : "#808080",
    "QW_2"  : "#808080",
    "NREM"  : "#D0D0D0",
    "NREM_3": "#D0D0D0",
    "IS"    : "#A0A0A0",
    "IS_4"  : "#A0A0A0",
    "REM"   : "white",
    "REM_5" : "white",
}

DISPLAY_RANGE_UV = 150

# =========================================================
# DOSSIERS PAR DÉFAUT
# =========================================================

DPATH_SCORING = r"\\10.69.168.1\crnldata\forgetting\Aurelie\MiniscopeOE_data\L1NDNF_mice\NV\Allocentric_task\Training\2026_02_23\SleepBefore"

DPATH_DAT = r"\\10.69.168.1\crnldata\forgetting\Aurelie\MiniscopeOE_data\L1NDNF_mice\NV\Allocentric_task\Training\2026_02_23\SleepBefore"

# =========================================================
# STOCKAGE GLOBAL
# =========================================================

selected = {
    "csv": None,
    "dat": None,
}

# =========================================================
# FILECHOOSER CSV
# =========================================================

print("=" * 60)
print("ÉTAPE 1 — Sélectionnez le fichier de scoring")
print("=" * 60)

fc_csv = FileChooser(
    DPATH_SCORING,
    filter_pattern=["*.csv", "*.xlsx"],
    title="<b> Fichier de scoring</b>"
)

display(fc_csv)

# =========================================================
# FILECHOOSER DAT
# =========================================================

print("=" * 60)
print("ÉTAPE 2 — Sélectionnez le fichier .dat")
print("=" * 60)

fc_dat = FileChooser(
    DPATH_DAT,
    filter_pattern=["*.dat"],
    title="<b> Fichier .dat OpenEphys</b>"
)

display(fc_dat)

# =========================================================
# FILTRES
# =========================================================

def highpass_filter(signal, fs, cutoff, order=8):
    """Retire le drift basse fréquence avant le bandpass."""
    nyq  = fs / 2
    b, a = butter(order, cutoff / nyq, btype="high")
    return filtfilt(b, a, signal)

def bandpass_filter(signal, fs, low, high, order=4):
    nyq  = fs / 2
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, signal)

# =========================================================
# FONCTION PRINCIPALE
# =========================================================

def generate_figure():

    csv_path = selected["csv"]
    dat_path = selected["dat"]

    if csv_path is None or dat_path is None:
        print(" En attente des deux fichiers...")
        return

    print("\n" + "=" * 60)
    print(" Génération de la figure")
    print("=" * 60)

    print(f"CSV : {csv_path}")
    print(f"DAT : {dat_path}")

    # =====================================================
    # 1. CHARGEMENT SCORING
    # =====================================================

    if csv_path.endswith(".csv"):
        df_score = pd.read_csv(csv_path)
    else:
        df_score = pd.read_excel(csv_path)

    df_score.columns = df_score.columns.str.strip().str.lower()

    print("\n Colonnes détectées :", df_score.columns.tolist())

    if all(col in df_score.columns for col in ["time", "duration", "label"]):
        df_score["start_time"] = df_score["time"]
        df_score["end_time"]   = df_score["time"] + df_score["duration"]

    elif all(col in df_score.columns for col in ["start_time", "end_time", "label"]):
        pass

    elif all(col in df_score.columns for col in ["start", "end", "label"]):
        df_score["start_time"] = df_score["start"]
        df_score["end_time"]   = df_score["end"]

    else:
        raise ValueError(
            " Format scoring non reconnu.\n"
            f"Colonnes trouvées : {df_score.columns.tolist()}"
        )

    print(" Labels détectés :", df_score["label"].unique())

    # =====================================================
    # 2. SAMPLE RATE
    # =====================================================

    datfilepath = Path(dat_path)
    oebin_file  = datfilepath.parent.parent.parent.parent.parent / "structure.oebin"

    samplerate = SAMPLERATE_DEFAULT
    numchannel = None
    bit_volts  = 0.195   # forcé car structure.oebin introuvable

    if oebin_file.exists():

        print(f"\n structure.oebin trouvé")

        with open(oebin_file) as f:
            metadata = json.load(f)

        samplerate = int(metadata["continuous"][0]["sample_rate"])
        numchannel = metadata["continuous"][0]["num_channels"]
        bit_volts  = metadata["continuous"][0]["channels"][0]["bit_volts"]

        print(f" Sample rate : {samplerate} Hz")
        print(f" Nb canaux   : {numchannel}")
        print(f" Conversion  : {bit_volts} µV/count")

    else:
        print("\n structure.oebin introuvable")
        print(f"→ bit_volts forcé : {bit_volts}")
        print(f"→ Sample rate défaut : {samplerate} Hz")

    # =====================================================
    # 3. CHARGEMENT DAT
    # =====================================================

    print("\n Chargement du .dat ...")

    raw = np.fromfile(dat_path, dtype="int16")

    if numchannel is None:

        duree_scoring = df_score["end_time"].max()

        for n in [32, 16, 8, 64]:
            if raw.size % n == 0:
                n_samples_test = raw.size // n
                duree_test     = n_samples_test / samplerate
                if abs(duree_test - duree_scoring) < duree_scoring * 0.1:
                    numchannel = n
                    print(f"→ Nb canaux détecté : {numchannel}")
                    break

        if numchannel is None:
            numchannel = 32
            print(f"→ Nb canaux par défaut : {numchannel}")

    DataRec      = raw.reshape(-1, numchannel)
    duree_reelle = DataRec.shape[0] / samplerate

    print(f" DAT chargé : {DataRec.shape}")
    print(f"⏱ Durée réelle : {duree_reelle:.1f} s")

    # =====================================================
    # 4. CHECK CANAUX
    # =====================================================

    for name, ch_idx in CHANNELS.items():
        if ch_idx >= DataRec.shape[1]:
            raise IndexError(f" Canal {name} hors limites")

    # =====================================================
    # 5. EXTRACTION FENÊTRE
    # =====================================================

    dur           = DURATION_TO_PLOT if DURATION_TO_PLOT is not None else duree_reelle
    duration_plot = min(dur, duree_reelle - START_TIME_PLOT)

    start_idx = int(START_TIME_PLOT * samplerate)
    end_idx   = min(int((START_TIME_PLOT + duration_plot) * samplerate), DataRec.shape[0])

    ds = DOWNSAMPLE

    print(f"\n Fenêtre affichée : {START_TIME_PLOT}s → {START_TIME_PLOT + duration_plot:.1f}s")

    # =====================================================
    # 6. FILTRAGE PAR CANAL + DOWNSAMPLE
    # =====================================================

    signals_ds = {}

    for name, ch_idx in CHANNELS.items():

        sig = (
            DataRec[start_idx:end_idx, ch_idx]
            .astype(float)
            * bit_volts
        )

        # --- ÉTAPE 1 : retrait du drift ---
        # PFC, CA1, RSC : highpass agressif à 5 Hz pour éliminer le fuseau
        # S1, EMG       : highpass léger à 1 Hz suffit
        if name in ["PFC", "CA1", "RSC"]:
            sig = highpass_filter(sig, samplerate, cutoff=5.0, order=8)
        else:
            sig = highpass_filter(sig, samplerate, cutoff=1.0, order=6)

        # --- ÉTAPE 2 : centrage résiduel ---
        sig = sig - np.median(sig)
        half = len(sig) // 2
        p99 = np.percentile(np.abs(sig), 95)
        if p99 > 0:
            sig = sig / p99 * 80

        # --- ÉTAPE 3 : bandpass spécifique au canal ---
        low, high = BANDPASS[name]
        sig = bandpass_filter(sig, samplerate, low, high)

        # --- ÉTAPE 4 : decimation ---
        sig_ds = decimate(sig, ds, zero_phase=True)

        signals_ds[name] = sig_ds

    # =====================================================
    # TEMPS DOWNSAMPLED
    # =====================================================

    npoints = len(signals_ds["PFC"])

    time = np.linspace(
        START_TIME_PLOT,
        START_TIME_PLOT + duration_plot,
        npoints
    )

    print(f" Points affichés : {npoints}")

    # =====================================================
    # 7. FIGURE
    # =====================================================

    signal_names = list(CHANNELS.keys())
    n_sig        = len(signal_names)

    fig, axes = plt.subplots(
        n_sig + 1, 1,
        figsize=(20, 10),
        sharex=True,
        gridspec_kw={"height_ratios": [1] * n_sig + [0.35]}
    )

    fig.patch.set_facecolor("white")

    # =====================================================
    # 8. TRACE SIGNAUX
    # =====================================================

    YLABEL = {
        "PFC": "LFP PFC\n1–50 Hz",
        "S1" : "LFP S1\n1–50 Hz",
        "CA1": "LFP CA1\n1–50 Hz",
        "RSC": "LFP RSC\n1–50 Hz",
        "EMG": "EMG\n30–120 Hz",
    }

    for i, name in enumerate(signal_names):

        ax  = axes[i]
        sig = signals_ds[name]

        ax.plot(
            time, sig,
            color=SIGNAL_COLORS[name],
            linewidth=0.35,
            alpha=0.9,
            antialiased=True,
            rasterized=True
        )

        ax.set_ylim(-DISPLAY_RANGE_UV, DISPLAY_RANGE_UV)

        ax.set_ylabel(
            YLABEL[name],
            rotation=0,
            fontsize=11,
            labelpad=55,
            weight="normal",
            color=SIGNAL_COLORS[name],
            va="center"
        )

        ax.set_yticks([-DISPLAY_RANGE_UV, 0, DISPLAY_RANGE_UV])
        ax.set_yticklabels(
            [f"−{DISPLAY_RANGE_UV}", "0", f"+{DISPLAY_RANGE_UV}"],
            fontsize=7,
            color="gray"
        )

        ax.tick_params(axis="y", length=2, width=0.5, color="gray")
        ax.set_facecolor("white")

        for spine in ["top", "right", "bottom"]:
            ax.spines[spine].set_visible(False)

        ax.spines["left"].set_visible(True)
        ax.spines["left"].set_color("lightgray")
        ax.spines["left"].set_linewidth(0.5)

    # =====================================================
    # UNITÉ µV
    # =====================================================

    fig.text(
        0.005, 0.55, "µV",
        va="center", ha="left",
        fontsize=10, color="gray",
        rotation=90
    )

    # =====================================================
    # 9. BARRE D'ÉCHELLE TEMPORELLE
    # =====================================================

    ax_last_sig = axes[n_sig - 1]
    scale_dur   = 60
    x_scale     = START_TIME_PLOT + duration_plot - 1.4 * scale_dur
    y_lim       = ax_last_sig.get_ylim()
    y_scale     = y_lim[0] + (y_lim[1] - y_lim[0]) * 0.15

    ax_last_sig.plot(
        [x_scale, x_scale + scale_dur],
        [y_scale, y_scale],
        color="black", linewidth=3, solid_capstyle="butt"
    )

    ax_last_sig.text(
        x_scale + scale_dur / 2,
        y_scale + (y_lim[1] - y_lim[0]) * 0.08,
        "1 min", ha="center", va="bottom", fontsize=10
    )

    # =====================================================
    # 10. HYPNOGRAMME
    # =====================================================

    ax_hyp = axes[-1]
    ax_hyp.set_facecolor("white")

    df_plot = df_score[
        (df_score["end_time"]   >= START_TIME_PLOT) &
        (df_score["start_time"] <= START_TIME_PLOT + duration_plot)
    ].copy()

    merged_states = []
    current_label = None
    current_start = None
    current_end   = None

    for _, row in df_plot.iterrows():

        label = str(row["label"]).strip()
        start = row["start_time"]
        end   = row["end_time"]

        if current_label is None:
            current_label = label
            current_start = start
            current_end   = end

        elif label == current_label:
            current_end = end

        else:
            merged_states.append({"label": current_label, "start": current_start, "end": current_end})
            current_label = label
            current_start = start
            current_end   = end

    merged_states.append({"label": current_label, "start": current_start, "end": current_end})

    for state in merged_states:
        x0    = max(state["start"], START_TIME_PLOT)
        x1    = min(state["end"],   START_TIME_PLOT + duration_plot)
        color = HYPNO_COLORS.get(state["label"], "red")
        ax_hyp.add_patch(Rectangle((x0, 0), x1 - x0, 1, facecolor=color, edgecolor="none"))

    legend_patches = [
        mpatches.Patch(facecolor=HYPNO_COLORS[lbl], edgecolor="black", linewidth=0.5, label=lbl)
        for lbl in ["AW", "QW", "NREM", "IS", "REM"]
    ]

    ax_hyp.legend(
        handles=legend_patches,
        loc="lower center",
        ncol=5,
        fontsize=10,
        frameon=False,
        bbox_to_anchor=(0.5, -0.8)
    )

    ax_hyp.set_ylim(0, 1)
    ax_hyp.set_yticks([])

    for spine in ax_hyp.spines.values():
        spine.set_visible(False)

    ax_hyp.set_xlim(START_TIME_PLOT, START_TIME_PLOT + duration_plot)
    ax_hyp.set_xticks([])

    # =====================================================
    # TITRE
    # =====================================================

    session_name = Path(dat_path).parent.parent.parent.parent.parent.name

    fig.suptitle(f"{session_name}", fontsize=14, weight="bold", y=0.995)

    plt.tight_layout(h_pad=0.05)

    # =====================================================
    # SAUVEGARDE
    # =====================================================

    plt.savefig(
        os.path.join(os.path.expanduser("~/Documents"), "Hypnogramme_PaperStyle.svg"),
        format="svg", bbox_inches="tight", facecolor="white"
    )

# =========================================================
# CALLBACKS
# =========================================================

def on_csv_selected(chooser):
    selected["csv"] = chooser.selected
    print(f" CSV sélectionné : {selected['csv']}")
    generate_figure()

def on_dat_selected(chooser):
    selected["dat"] = chooser.selected
    print(f" DAT sélectionné : {selected['dat']}")
    generate_figure()

fc_csv.register_callback(on_csv_selected)
fc_dat.register_callback(on_dat_selected)